# L1. The On-Device Memory Problem

Your AI has a model on the device and it can search vectors, but it has nowhere to **remember** anything. In this short notebook you set up the fix: a **Qdrant Edge** memory store that lives inside your process, on your disk.

That store is a **shard**, the unit a Qdrant collection is made of. On a server a collection spreads across many shards; Edge gives you exactly one, running in-process. The full build starts in L2.

## 1. Configure the memory store

In [6]:
from qdrant_edge import EdgeConfig, EdgeVectorParams, Distance

config = EdgeConfig(
    vectors={
        "text": EdgeVectorParams(size=768, distance=Distance.Cosine),
    }
)
print("One named vector: text, 768-d, cosine distance")

One named vector: text, 768-d, cosine distance


## 2. Create it

In [7]:
from pathlib import Path
from qdrant_edge import EdgeShard

SHARD_DIR = "./first_shard"
Path(SHARD_DIR).mkdir(parents=True, exist_ok=True)
shard = EdgeShard.create(SHARD_DIR, config)
print("EdgeShard created at", SHARD_DIR)

EdgeShard created at ./first_shard


## 3. Write the first memory

In [8]:
from qdrant_edge import Point, UpdateOperation
from helper import embed_text

note = "Great little coffee place on 5th with outdoor seating and fast wifi"
vector = embed_text([note])[0]

shard.update(UpdateOperation.upsert_points([
    Point(id=0, vector={"text": vector}, payload={"note": note}),
]))
print("Memories stored:", shard.info().points_count)

Memories stored: 1


## 4. The payoff: a memory store that is just files

In [9]:
shard.close()

files = [p for p in Path(SHARD_DIR).rglob("*") if p.is_file()]
for p in sorted(files)[:8]:
    print("  ", p.relative_to(SHARD_DIR))
print(f"\n{len(files)} plain files in a local folder: your AI's memory, no server")

   edge_config.json
   segments/35d29b91-b6e8-489b-95cf-93cb8b1020a3/mutable_id_tracker.mappings
   segments/35d29b91-b6e8-489b-95cf-93cb8b1020a3/mutable_id_tracker.versions
   segments/35d29b91-b6e8-489b-95cf-93cb8b1020a3/payload_index/config.json
   segments/35d29b91-b6e8-489b-95cf-93cb8b1020a3/payload_storage/bitmask.dat
   segments/35d29b91-b6e8-489b-95cf-93cb8b1020a3/payload_storage/config.json
   segments/35d29b91-b6e8-489b-95cf-93cb8b1020a3/payload_storage/gaps.dat
   segments/35d29b91-b6e8-489b-95cf-93cb8b1020a3/payload_storage/page_0.dat

18 plain files in a local folder: your AI's memory, no server


In [10]:
import shutil
shutil.rmtree(SHARD_DIR, ignore_errors=True)
print("Cleaned up", SHARD_DIR)

Cleaned up ./first_shard
